# 24 — Consensus modelleri: iç ve dış test değerlendirmesi

Bu notebook, `23_finetuning_with_3_consensus.ipynb` ile eğitilen BERT, DistilBERT, RoBERTa ve FinBERT modellerini değerlendirir.

- İç test: üçlü-consensus eğitim akışında ayrılan Twitter `internal_test.csv`
- Dış test: S&P 500, sentetik finans haberleri ve Reuters 5000
- Bu notebook yeniden eğitim yapmaz; yalnızca `final_model` klasörlerini yükler.
- Her model/veri kümesi için tahminler, sınıflandırma raporu ve confusion matrix kaydedilir.

In [1]:
from thesis_utils import DATA_ROOT, paths
from pathlib import Path
import json, re, inspect
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from thesis_utils.data_io import find_header_row, read_raw_table_no_header, make_unique_columns, normalize_column_name
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report, confusion_matrix

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
LABELS = list(LABEL2ID)
MAX_LENGTH = 128
MODEL_ROOT = paths.MODEL_CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1'
RESULTS_ROOT = paths.MODEL_RESULTS_ROOT / 'consensus_twitter_phrasebank_allagree_v1' / 'evaluation'
INTERNAL_PATH = DATA_ROOT / 'splits' / 'consensus_twitter_phrasebank_allagree_v1' / 'internal_test.csv'
SP500_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR
SYNTHETIC_DIR = paths.SYNTHETIC_NEWS_DIR
REUTERS_DIR = paths.REUTERS_ANNOTATION_DIR
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CONFIGS = [
    {'model_key':'bert', 'display_name':'BERT-base-uncased'},
    {'model_key':'distilbert', 'display_name':'DistilBERT-base-uncased'},
    {'model_key':'roberta', 'display_name':'RoBERTa-base'},
    {'model_key':'finbert', 'display_name':'FinBERT'},
]
print('Model root:', MODEL_ROOT)
print('Results root:', RESULTS_ROOT)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

Model root: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\consensus_twitter_phrasebank_allagree_v1
Results root: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\outputs\finetuned_model_results\consensus_twitter_phrasebank_allagree_v1\evaluation
Device: cpu


In [2]:
def clean_text(s):
    return s.astype(str).str.strip().replace({'': np.nan, 'nan': np.nan, 'None': np.nan})

def normalize_columns(df):
    out = df.copy()
    out.columns = [re.sub(r'[^a-z0-9]+', '_', str(c).strip().lower().replace('ı','i')).strip('_') for c in out.columns]
    return out

def pick_text(df):
    for c in ['input_text','text_en','text','source_text','headline','title']:
        if c in df.columns and clean_text(df[c]).notna().any(): return c
    raise ValueError(f'Metin kolonu bulunamadı: {df.columns.tolist()}')

def make_eval_df(df, dataset_name, gold_candidates):
    df = normalize_columns(df)
    text_col = pick_text(df)
    df['eval_text'] = clean_text(df[text_col])
    df['gold_label'] = pd.NA
    for c in gold_candidates:
        if c in df.columns: df['gold_label'] = df['gold_label'].fillna(clean_text(df[c]).str.lower())
    df = df[df['eval_text'].notna() & df['gold_label'].isin(LABELS)].copy()
    # Dış testte aynı metin farklı annotation_id ile geçebileceği için
    # metin bazlı dedup yapılmaz; yalnızca Reuters'ta annotation_id korunur.
    df = df.reset_index(drop=True)
    df['gold_id'] = df['gold_label'].map(LABEL2ID).astype(int)
    if 'sample_id' not in df.columns: df.insert(0, 'sample_id', [f'{dataset_name}_{i:06d}' for i in range(len(df))])
    return df

def read_table(path):
    raw = read_raw_table_no_header(path).dropna(how='all').copy()
    header_idx = find_header_row(raw)
    if header_idx is None:
        for i, row in raw.head(20).iterrows():
            vals = {re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower().replace('ı','i')).strip('_') for x in row.tolist()}
            if 'annotation_id' in vals and vals & {'text_en','text','source_text','headline','chatgpt_label','final_label'}:
                header_idx = i; break
    if header_idx is None:
        df = pd.read_excel(path, dtype=object) if path.suffix.lower() == '.xlsx' else pd.read_csv(path, dtype=object, encoding='utf-8-sig', engine='python')
    else:
        df = raw.iloc[header_idx + 1:].copy()
        df.columns = make_unique_columns(raw.iloc[header_idx].tolist())
    return df.dropna(how='all').dropna(axis=1, how='all').copy()

def load_internal():
    if not INTERNAL_PATH.exists(): raise FileNotFoundError(f'İç test bulunamadı: {INTERNAL_PATH}')
    return make_eval_df(pd.read_csv(INTERNAL_PATH), 'internal_test', ['label'])

def load_sp500():
    files = sorted([p for p in SP500_DIR.rglob('SP500_annotation_batch_*.xlsx') if not p.name.startswith('~$')])
    if not files: raise FileNotFoundError(f'S&P 500 batch bulunamadı: {SP500_DIR}')
    frames=[]
    for p in files:
        try: frames.append(read_table(p))
        except Exception as e: print('S&P5k atlandı:', p.name, e)
    return make_eval_df(pd.concat(frames, ignore_index=True), 'sp500_external', ['final_label','chatgpt_label'])

def load_synthetic():
    files = sorted([p for pat in ['positive_batch_*.xlsx','negative_batch_*.xlsx','neutral_batch_*.xlsx'] for p in SYNTHETIC_DIR.glob(pat) if not p.name.startswith('~$')])
    if not files: raise FileNotFoundError(f'Sentetik batch bulunamadı: {SYNTHETIC_DIR}')
    return make_eval_df(pd.concat([read_table(p) for p in files], ignore_index=True), 'synthetic_external', ['label'])

def load_reuters():
    # CSV ara çıktılarında ChatGPT etiketi çoğunlukla boş; etiketli XLSX
    # batchleri tercih ediyoruz. XLSX yoksa CSV'ye düşülür.
    files = sorted([p for p in REUTERS_DIR.rglob('*annotation_batch_*.xlsx') if not p.name.startswith('~$')])
    if not files: files = sorted([p for p in REUTERS_DIR.rglob('*annotation_batch_*.csv') if not p.name.startswith('~$')])
    if not files: raise FileNotFoundError(f'Reuters batch bulunamadı: {REUTERS_DIR}')
    frames=[]
    for p in files:
        try: frames.append(read_table(p))
        except Exception as e: print('Reuters atlandı:', p.name, e)
    df = pd.concat(frames, ignore_index=True)
    if 'annotation_id' in df.columns: df = df.drop_duplicates('annotation_id')
    # Batch klasöründe yalnızca kısmi ChatGPT etiketleri varsa,
    # projedeki temizlenmiş Reuters consensus çıktısını kullan.
    clean_path = DATA_ROOT / 'processed' / 'gemini_consensus_2024' / 'reuters_annotation_clean_consensus.csv'
    candidate = make_eval_df(df, 'reuters_external', ['final_label','chatgpt_label'])
    if len(candidate) < 3000 and clean_path.exists():
        candidate = make_eval_df(pd.read_csv(clean_path), 'reuters_external', ['chatgpt_label','gemini_label'])
    return candidate

eval_sets = {'internal_test': load_internal(), 'sp500_external': load_sp500(), 'synthetic_external': load_synthetic(), 'reuters_external': load_reuters()}
for name, df in eval_sets.items(): print(name, len(df), df['gold_label'].value_counts().to_dict())

internal_test 1630 {'neutral': 893, 'positive': 423, 'negative': 314}
sp500_external 1060 {'positive': 398, 'neutral': 339, 'negative': 323}
synthetic_external 3000 {'negative': 1000, 'neutral': 1000, 'positive': 1000}
reuters_external 3842 {'positive': 1615, 'negative': 1176, 'neutral': 1051}


In [3]:
def tokenize_df(df, tokenizer):
    temp = df[['eval_text','gold_id']].rename(columns={'gold_id':'labels'}).copy()
    ds = Dataset.from_pandas(temp.reset_index(drop=True))
    ds = ds.map(lambda b: tokenizer(b['eval_text'], truncation=True, padding='max_length', max_length=MAX_LENGTH), batched=True)
    keep = ['input_ids','attention_mask','labels']
    if 'token_type_ids' in ds.column_names: keep.append('token_type_ids')
    ds.set_format(type='torch', columns=keep)
    return ds

def tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    if 'processing_class' in sig.parameters: return {'processing_class': tokenizer}
    if 'tokenizer' in sig.parameters: return {'tokenizer': tokenizer}
    return {}

def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True); e = np.exp(x); return e / e.sum(axis=1, keepdims=True)

def json_safe(value):
    # NumPy/pandas scalar ve array değerlerini standart Python tiplerine çevirir.
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, np.integer): return int(value)
    if isinstance(value, np.floating): return float(value)
    if isinstance(value, np.bool_): return bool(value)
    return value

def evaluate_model_dataset(model_key, display_name, dataset_name, df):
    final_dir = MODEL_ROOT / model_key / 'final_model'
    if not final_dir.exists(): raise FileNotFoundError(f'{display_name} final model bulunamadı: {final_dir}')
    out_dir = RESULTS_ROOT / model_key / dataset_name; out_dir.mkdir(parents=True, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(final_dir)
    model = AutoModelForSequenceClassification.from_pretrained(final_dir)
    model.config.id2label, model.config.label2id = ID2LABEL.copy(), LABEL2ID.copy()
    ds = tokenize_df(df, tokenizer)
    eval_args = TrainingArguments(output_dir=str(out_dir / 'runtime'), per_device_eval_batch_size=64, report_to='none')
    trainer = Trainer(model=model, args=eval_args, **tokenizer_kwargs(tokenizer))
    pred = trainer.predict(ds)
    probs = softmax(pred.predictions); y_true = df['gold_id'].to_numpy(dtype=int); y_pred = probs.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0,1,2], zero_division=0)
    summary = {'model_key':model_key, 'model':display_name, 'dataset':dataset_name, 'n_eval':len(df), 'accuracy':accuracy_score(y_true,y_pred), 'precision_macro':precision.mean(), 'recall_macro':recall.mean(), 'f1_macro':f1.mean(), 'f1_weighted':f1_score(y_true,y_pred,average='weighted'), 'f1_negative':f1[0], 'f1_neutral':f1[1], 'f1_positive':f1[2], 'support_negative':support[0], 'support_neutral':support[1], 'support_positive':support[2]}
    pred_df = df.copy(); pred_df['predicted_label'] = [ID2LABEL[int(x)] for x in y_pred]; pred_df['confidence'] = probs.max(axis=1)
    for i, label in ID2LABEL.items(): pred_df[f'prob_{label}'] = probs[:,i]
    pred_df.to_csv(out_dir / 'predictions.csv', index=False, encoding='utf-8-sig')
    report = pd.DataFrame(classification_report(y_true, y_pred, labels=[0,1,2], target_names=LABELS, output_dict=True, zero_division=0)).T
    report.to_csv(out_dir / 'classification_report.csv', encoding='utf-8-sig')
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2]); pd.DataFrame(cm, index=[f'true_{x}' for x in LABELS], columns=[f'pred_{x}' for x in LABELS]).to_csv(out_dir / 'confusion_matrix.csv', encoding='utf-8-sig')
    row_sum = cm.sum(axis=1, keepdims=True); norm = np.divide(cm, row_sum, out=np.zeros_like(cm,dtype=float), where=row_sum!=0)
    pd.DataFrame(norm, index=[f'true_{x}' for x in LABELS], columns=[f'pred_{x}' for x in LABELS]).to_csv(out_dir / 'confusion_matrix_normalized.csv', encoding='utf-8-sig')
    (out_dir / 'metrics.json').write_text(json.dumps(json_safe(summary), indent=2, ensure_ascii=False), encoding='utf-8')
    del trainer, model, tokenizer, ds
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

all_results=[]
for cfg in MODEL_CONFIGS:
    for dataset_name, df in eval_sets.items():
        print(f'Evaluate: {cfg["display_name"]} | {dataset_name}')
        all_results.append(evaluate_model_dataset(cfg['model_key'], cfg['display_name'], dataset_name, df))
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(RESULTS_ROOT / 'all_models_all_datasets_summary.csv', index=False, encoding='utf-8-sig')
display(summary_df[['model','dataset','n_eval','accuracy','precision_macro','recall_macro','f1_macro','f1_weighted']].round(4))

Evaluate: BERT-base-uncased | internal_test


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/1630 [00:00<?, ? examples/s]

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Evaluate: BERT-base-uncased | sp500_external


...

Evaluate: FinBERT | reuters_external


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/3842 [00:00<?, ? examples/s]

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,model,dataset,n_eval,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,BERT-base-uncased,internal_test,1630,0.9264,0.9092,0.9268,0.9172,0.9269
1,BERT-base-uncased,sp500_external,1060,0.8075,0.8067,0.8068,0.8066,0.8074
2,BERT-base-uncased,synthetic_external,3000,0.9817,0.9818,0.9817,0.9816,0.9816
3,BERT-base-uncased,reuters_external,3842,0.7827,0.7951,0.7915,0.7828,0.7886
4,DistilBERT-base-uncased,internal_test,1630,0.9080,0.8902,0.9011,0.8954,0.9084
5,DistilBERT-base-uncased,sp500_external,1060,0.7811,0.7818,0.7786,0.7797,0.7808
6,DistilBERT-base-uncased,synthetic_external,3000,0.9687,0.9699,0.9687,0.9686,0.9686
7,DistilBERT-base-uncased,reuters_external,3842,0.7530,0.7700,0.7622,0.7539,0.7599
8,RoBERTa-base,internal_test,1630,0.9515,0.9383,0.9553,0.9463,0.9518
9,RoBERTa-base,sp500_external,1060,0.8330,0.8343,0.8304,0.8319,0.8328
